# 06. <span style="color:#7CDBA0">Dataset</span> и <span style="color:#6CB6FF">DataLoader</span> — ч.3 (практика)

Видео: [6. Подготовка данных ч.3. <span style="color:#7CDBA0">Dataset</span> и <span style="color:#6CB6FF">DataLoader</span>](https://www.youtube.com/watch?v=GKRZAflTnQI&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=7)

## Зачем

Видео 5 — свой <span style="color:#7CDBA0">Dataset</span> «с нуля». Здесь два продолжения автора:

1. **<span style="color:#E6C07B">ImageFolder</span>** — та же раскладка «папка = класс», но одной строкой (то, что мы руками писали как `FruitsDataset`).
2. **Регрессия** — картинки + json с числами (не классами). У автора квадрат → центр `(y, x)`. У нас **круг** → центр и радиус `(cx, cy, r)` формы `(3,)`.

**Как смотреть.** Видео короткое (~8 мин) — можно целиком. Удобнее кусками по таблице → пауза → блок.

| блок | о чём | где в видео |
|:---|:---|:---|
| 1 | <span style="color:#E6C07B">ImageFolder</span> на фруктах | 0:00–2:38 |
| 2 | свой <span style="color:#7CDBA0">CirclesDataset</span> (регрессия) | 2:38–7:47 |

**Как решать.** Замени `...` на код → запусти проверку. Решения — в `06_dataset_dataloader_3_решения.ipynb`, только после своей попытки.


In [ ]:
import json
import os
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Grayscale, ToTensor

torch.manual_seed(0)

# ImageFolder по умолчанию открывает картинку как RGB → ToTensor даёт (3, H, W).
# Наши фрукты серые — как в 05 оставляем 1 канал.
GRAY_TENSOR = Compose([Grayscale(num_output_channels=1), ToTensor()])


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"


## 0. Данные

Ячейка ниже создаёт `_data/` рядом с ноутбуком:

```text
_data/
├── fruits/
│   ├── train/  apple|pear|plum  (+ мусорные txt — ImageFolder их игнорит)
│   └── test/   apple|pear|plum
└── circles/
    ├── img_000.png … img_199.png   # 64×64, серый фон + светлый круг
    └── labels.json                 # {"img_000.png": [cx, cy, r], ...}
```

Фрукты — как в `05` (проверка `ImageFolder`). Круги — мини-версия регрессии автора (у него 100 000 квадратов; нам хватит 200).


In [ ]:
DATA_ROOT = os.path.join(".", "_data")
FRUITS_TRAIN = os.path.join(DATA_ROOT, "fruits", "train")
FRUITS_TEST = os.path.join(DATA_ROOT, "fruits", "test")
CIRCLES_DIR = os.path.join(DATA_ROOT, "circles")
LABELS_PATH = os.path.join(CIRCLES_DIR, "labels.json")


def _write_fruit_png(path, value):
    arr = np.full((8, 8), value, dtype=np.uint8)
    Image.fromarray(arr, mode="L").save(path)


def _make_fruits():
    layout = {
        "train": {"apple": 12, "pear": 10, "plum": 3},
        "test": {"apple": 4, "pear": 4, "plum": 4},
    }
    brightness = {"apple": 40, "pear": 120, "plum": 200}
    for split, classes in layout.items():
        for cls, n in classes.items():
            d = os.path.join(DATA_ROOT, "fruits", split, cls)
            os.makedirs(d, exist_ok=True)
            for i in range(n):
                _write_fruit_png(
                    os.path.join(d, f"{cls}_{i:03d}.png"), brightness[cls]
                )
        # мусор, как в 05 — ImageFolder не должен считать это классом/картинкой
        open(os.path.join(DATA_ROOT, "fruits", split, "README.txt"), "w", encoding="utf-8").write("skip\n")
    open(
        os.path.join(DATA_ROOT, "fruits", "train", "pear", "notes.txt"),
        "w",
        encoding="utf-8",
    ).write("not an image\n")


def _draw_circle(cx, cy, r, size=64):
    img = Image.new("L", (size, size), color=30)
    draw = ImageDraw.Draw(img)
    # bbox круга; outline+width даёт видимое кольцо/диск
    box = [cx - r, cy - r, cx + r, cy + r]
    draw.ellipse(box, fill=220)
    return img


def _make_circles(n=200, size=64, seed=0):
    os.makedirs(CIRCLES_DIR, exist_ok=True)
    rng = np.random.default_rng(seed)
    labels = {}
    for i in range(n):
        name = f"img_{i:03d}.png"
        r = int(rng.integers(6, 18))
        cx = int(rng.integers(r + 1, size - r - 1))
        cy = int(rng.integers(r + 1, size - r - 1))
        _draw_circle(cx, cy, r, size=size).save(os.path.join(CIRCLES_DIR, name))
        labels[name] = [float(cx), float(cy), float(r)]
    with open(LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(labels, f, indent=2)
    return n


if not os.path.isdir(FRUITS_TRAIN):
    _make_fruits()
if not os.path.isfile(LABELS_PATH):
    _make_circles()

print("fruits train:", FRUITS_TRAIN)
print("circles:", CIRCLES_DIR, "labels:", LABELS_PATH)


## 1. <span style="color:#E6C07B">ImageFolder</span> (видео ~0:00–2:38)

Готовый датасет из `torchvision` для раскладки:

```text
root/
  class_a/  *.png
  class_b/  *.png
```

Одной строкой: `ImageFolder(root)`.

| что | у нашего `FruitsDataset` (05) | у <span style="color:#E6C07B">ImageFolder</span> |
|:---|:---|:---|
| список классов | сами собирали + `sorted` | `.classes` (уже отсортирован) |
| имя → номер | `.class_to_idx` | `.class_to_idx` |
| `__getitem__` | тензор `(1, H, W)` + `int64` метка | по умолчанию **PIL.Image** + `int` метка |
| мусорные txt | фильтровали сами | игнорирует не-картинки |

Без `transform` батч из PIL не склеится. Для батча нужен тензор. Готовый `GRAY_TENSOR` = `Grayscale(1)` + `ToTensor()`: канал один (как в `FruitsDataset`), значения в `[0, 1]`. Без `Grayscale` `ImageFolder` часто отдаёт RGB → форма `(3, H, W)`. Подробные transforms — видео 7.

**Подсказка:** уже импортированы `ImageFolder` и `GRAY_TENSOR`.


In [ ]:
# 1.1 Создай ImageFolder на train и на test.
# Ожидаемый вывод (смысл):
#   classes: ['apple', 'pear', 'plum']
#   class_to_idx: {'apple': 0, 'pear': 1, 'plum': 2}
#   len train / test: 25 / 12

train_folder = ...
test_folder = ...

print("classes:", train_folder.classes)
print("class_to_idx:", train_folder.class_to_idx)
print("len train / test:", len(train_folder), "/", len(test_folder))


In [ ]:
need(train_folder.classes == ["apple", "pear", "plum"], f"classes={train_folder.classes}")
need(
    train_folder.class_to_idx == {"apple": 0, "pear": 1, "plum": 2},
    f"class_to_idx={train_folder.class_to_idx}",
)
need(train_folder.class_to_idx == test_folder.class_to_idx, "train/test class_to_idx должны совпадать")
need(len(train_folder) == 25, f"len(train)={len(train_folder)}, ждали 25 (txt не считаются)")
need(len(test_folder) == 12, f"len(test)={len(test_folder)}, ждали 12")
img0, y0 = train_folder[0]
need(isinstance(img0, Image.Image), f"без transform __getitem__ даёт PIL, получили {type(img0)}")
need(isinstance(y0, int), f"метка должна быть int, получили {type(y0)}")
print("1.1 — верно")


In [ ]:
# 1.2 Тот же ImageFolder, но с transform=GRAY_TENSOR, чтобы DataLoader собрал батч.
# Ожидаемый вывод:
#   x shape/dtype: torch.Size([1, 8, 8]) torch.float32
#   batch X/y: torch.Size([8, 1, 8, 8]) torch.Size([8])

train_folder_t = ...
test_folder_t = ...

# train → train/val; test не трогаем (как у автора)
gen = torch.Generator().manual_seed(0)
train_part, val_part = ...

train_loader = ...
val_loader = ...
test_loader = ...

x0, y0 = train_folder_t[0]
xb, yb = next(iter(train_loader))
print("x shape/dtype:", x0.shape, x0.dtype)
print("batch X/y:", xb.shape, yb.shape)
print("sizes:", len(train_part), len(val_part), len(test_folder_t))


In [ ]:
need_tensor(x0, "x0")
need(tuple(x0.shape) == (1, 8, 8) and x0.dtype == torch.float32, f"x0: {tuple(x0.shape)} {x0.dtype}")
need(0.0 <= float(x0.min()) and float(x0.max()) <= 1.0, "ToTensor даёт диапазон [0, 1]")
need(len(train_part) == 20 and len(val_part) == 5, f"split {len(train_part)}/{len(val_part)}")
need_tensor(xb, "xb")
need(tuple(xb.shape) == (8, 1, 8, 8), f"xb.shape={tuple(xb.shape)}")
need(tuple(yb.shape) == (8,) and yb.dtype == torch.int64, f"yb: {tuple(yb.shape)} {yb.dtype}")
need(isinstance(train_loader.sampler, torch.utils.data.RandomSampler), "train: shuffle=True")
need(isinstance(val_loader.sampler, torch.utils.data.SequentialSampler), "val: shuffle=False")
need(isinstance(test_loader.sampler, torch.utils.data.SequentialSampler), "test: shuffle=False")
print("1.2 — верно")


**Зачем тогда писали `FruitsDataset` вручную?** Чтобы понять `__init__` / `__len__` / `__getitem__`. `ImageFolder` — сокращение *только* для «папка = класс». Клики, json с координатами, CSV — снова свой класс.


## 2. Регрессия: <span style="color:#7CDBA0">CirclesDataset</span> (видео ~2:38–7:47)

У автора: квадрат 15×15 на картинке 64×64, в json — центр `[y, x]`, ответ формы `(2,)`.

У нас: **круг** случайного радиуса, в `labels.json` — `[cx, cy, r]`, ответ формы `(3,)`.

```text
circles/
  img_000.png
  ...
  labels.json   ← {"img_000.png": [cx, cy, r], ...}
```

Каркас тот же, что у `FruitsDataset` / кликов:

1. `__init__` — список имён картинок + словарь разметки (json.load), **не** грузить все пиксели.
2. `__len__` — число картинок.
3. `__getitem__(i)` — открыть одну картинку, достать `[cx, cy, r]`, вернуть тензоры.

**Формы:** `x` → `(1, 64, 64)` `float32` в `[0, 1]`; `y` → `(3,)` `float32`.

**Подсказки:**
- список файлов: `os.listdir` → оставить только `.png` (или удалить `labels.json` из списка, как автор);
- json: `with open(...) as f: labels = json.load(f)`;
- картинка: `Image.open(path)` → `np.array(...)` → `torch.from_numpy(...).float() / 255` → `.unsqueeze(0)`;
- ответ: `torch.tensor(labels[name], dtype=torch.float32)`.


In [ ]:
# 2.1 Допиши CirclesDataset.
# Ожидаемый вывод:
#   len: 200
#   x: torch.Size([1, 64, 64]) torch.float32
#   y: torch.Size([3]) torch.float32   # [cx, cy, r]


class CirclesDataset(Dataset):
    def __init__(self, root):
        self.root = root
        ...
        ...
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...
        ...
        return x, y


circles_ds = CirclesDataset(CIRCLES_DIR)
x, y = circles_ds[0]
print("len:", len(circles_ds))
print("x:", x.shape, x.dtype)
print("y:", y.shape, y.dtype, y.tolist())


In [ ]:
need(len(circles_ds) == 200, f"len={len(circles_ds)}")
need_tensor(x, "x")
need_tensor(y, "y")
need(tuple(x.shape) == (1, 64, 64) and x.dtype == torch.float32, f"x: {tuple(x.shape)} {x.dtype}")
need(tuple(y.shape) == (3,) and y.dtype == torch.float32, f"y: {tuple(y.shape)} {y.dtype}")
need(0.0 <= float(x.min()) and float(x.max()) <= 1.0, "x должен быть в [0, 1]")
# сверим с json для img_000.png (после sort это names[0])
with open(LABELS_PATH, encoding="utf-8") as f:
    _lab = json.load(f)
need(circles_ds.names[0] == "img_000.png", f"names[0]={circles_ds.names[0]}")
need(y.tolist() == _lab["img_000.png"], f"y={y.tolist()} != {_lab['img_000.png']}")
print("2.1 — верно")


### 2.2 Глазами: метка совпадает с кругом

Как у автора с красной точкой на квадрате — нарисуй окружность по `(cx, cy, r)` поверх картинки. Если класс верный, кольцо ляжет на светлый круг.


In [ ]:
def show_circle(ds, idx=0):
    img, target = ds[idx]
    cx, cy, r = target.tolist()
    # img: (1, H, W) → для imshow (H, W)
    plt.figure(figsize=(3, 3))
    plt.imshow(img.squeeze(0).numpy(), cmap="gray", vmin=0, vmax=1)
    circ = plt.Circle((cx, cy), r, fill=False, color="red", linewidth=2)
    plt.gca().add_patch(circ)
    plt.scatter([cx], [cy], c="red", s=20)
    plt.title(f"idx={idx}  cx={cx:.0f} cy={cy:.0f} r={r:.0f}")
    plt.axis("off")
    plt.show()


show_circle(circles_ds, 0)
show_circle(circles_ds, 7)


In [ ]:
# 2.3 random_split + три DataLoader (как у автора на квадратах).
# Пропорции на 200: train 140 / val 30 / test 30, batch_size=16.
# Ожидаемый вывод:
#   sizes: 140 30 30
#   batch X/y: torch.Size([16, 1, 64, 64]) torch.Size([16, 3])

gen = torch.Generator().manual_seed(0)
circ_train, circ_val, circ_test = ...

circ_train_loader = ...
circ_val_loader = ...
circ_test_loader = ...

cxb, cyb = next(iter(circ_train_loader))
print("sizes:", len(circ_train), len(circ_val), len(circ_test))
print("batch X/y:", cxb.shape, cyb.shape)
print("last train batch len:", len(circ_train) % 16)  # 140 % 16 = 12 → короткий хвост


In [ ]:
need(
    (len(circ_train), len(circ_val), len(circ_test)) == (140, 30, 30),
    f"sizes={(len(circ_train), len(circ_val), len(circ_test))}",
)
need_tensor(cxb, "cxb")
need_tensor(cyb, "cyb")
need(tuple(cxb.shape) == (16, 1, 64, 64), f"cxb.shape={tuple(cxb.shape)}")
need(tuple(cyb.shape) == (16, 3), f"cyb.shape={tuple(cyb.shape)}")
need(isinstance(circ_train_loader.sampler, torch.utils.data.RandomSampler), "train shuffle")
need(isinstance(circ_val_loader.sampler, torch.utils.data.SequentialSampler), "val no shuffle")
need(isinstance(circ_test_loader.sampler, torch.utils.data.SequentialSampler), "test no shuffle")
# короткий последний батч train: 140 = 8*16 + 12
n_full = 0
last = None
for batch in circ_train_loader:
    n_full += 1
    last = batch[0].shape[0]
need(n_full == 9 and last == 12, f"ждали 8 полных + хвост 12, получили n={n_full} last={last}")
print("2.3 — верно")


## 3. Что унести из видео 6

| инструмент | когда |
|:---|:---|
| свой <span style="color:#7CDBA0">Dataset</span> | любой формат: json, CSV, клики, координаты |
| <span style="color:#E6C07B">ImageFolder</span> | только «папка = класс» для картинок |

`ImageFolder` не заменяет понимание `__getitem__` — он сам внутри делает то же самое.

Для регрессии ответ — **числа** `(B, K)`, не номер класса. Здесь `K=3` (`cx, cy, r`); у автора `K=2`.

### Чеклист

1. `ImageFolder(root, transform=...)` → `.classes`, `.class_to_idx`, `(картинка, int)`.
2. Без тензорного `transform` — PIL; для серого батча — `Grayscale(1) + ToTensor` (у нас `GRAY_TENSOR`).
3. Регрессия: json-словарь имя→числа; в `__getitem__` читаем один файл.
4. `random_split` + три `DataLoader`: train shuffle, val/test без; последний батч может быть короче.

**Дальше:** раздел «Видео 6» в `project/fashion_mnist.ipynb` (Fashion через `ImageFolder` + те же круги в проекте). Потом видео 7 — transforms.
